# Imports

In [ ]:
import squad
import promptEngineer
from smolLM import SmolLM
from llama3 import Llama3API
from llama3 import Llama3Local
from gemini import GeminiAPI
from Gemma import Gemma
from llama3_small import Llama3B
from phi import Phi
from evaluation import Evaluater
from llama3_1B import Llama1B
from FlanT5 import FlanT5

import numpy as np
import time

# Loading Data

In [ ]:
data = squad.SQUAD()

i = 5
(question, context, answer) = data[i]

print("Question: {}".format(question))
print("Context: {}".format(context))
print("Answer: {}".format(answer))

# Convert to Prompts

In [ ]:
prompt_engineer = promptEngineer.PromptEngineer()

contexts = data.get_contexts()

long_contexts = []
N = len(contexts)

for i in range(len(contexts)):
    if i > 0 and i < N - 1:
        C = contexts[i-1] + "[[NEW CONTEXT]]" +  contexts[i] + "[[NEW CONTEXT]]" + contexts[i + 1]
    else:
        C = contexts[i]

    long_contexts.append(C)


prompts_with_context = prompt_engineer.construct_prompts_with_context(data.get_questions(), data.get_contexts())
prompts_with_long_context = prompt_engineer.construct_prompts_with_context(data.get_questions(), long_contexts)
prompts_without_context = prompt_engineer.construct_prompts_without_context(data.get_questions())

i = 29

print(prompts_with_context[i])
print("\n")
print(prompts_without_context[i])

true_answers = data.get_answers()

In [ ]:
evaluater = Evaluater()

In [ ]:
def test_model(model, sample_indices, test_prompts, batch_size = 6):
    prompts = [test_prompts[index] for index in sample_indices]
    true_answers_slice = [true_answers[index] for index in sample_indices]

    N = len(sample_indices)

    assert(len(prompts) == len(sample_indices))
    

    print("\n")

    mean_jaccard = 0
    mean_recall = 0
    mean_precision = 0
    mean_f1 = 0
    mean_edit_dist = 0
    mean_exact_match = 0

    num_runs = np.int32(np.ceil(N / batch_size))

    time_start = time.time()

    cnt = 0

    for i in range(0, num_runs):
        i_start = batch_size * i
        i_end = batch_size * (i + 1)
        prompts_batch = prompts[i_start:i_end]
        predicted_answers_batch, predicted_answers_extracted_batch = model.answer(prompts_batch)
        true_answers_slice_batch = true_answers_slice[i_start:i_end]
        for (prompt, pred_answer, pred_answer_extract, true_answer) in zip(prompts_batch, predicted_answers_batch, predicted_answers_extracted_batch, true_answers_slice_batch):
            print("Prompt: {}\n".format(prompt))
            print("Pred Answer Extract: {}\n".format(pred_answer_extract))
            print("True Answer: {}\n".format(true_answer))

            jaccard = evaluater.jaccard_similarity(true_answer, pred_answer_extract)
            recall = evaluater.recall(true_answer, pred_answer_extract)
            precision = evaluater.precision(true_answer, pred_answer_extract)
            f1 = evaluater.f1(true_answer, pred_answer_extract)
            edit_dist = evaluater.edit_distance(true_answer, pred_answer_extract)
            exact_match = evaluater.exact_match(true_answer, pred_answer_extract)

            mean_jaccard += jaccard
            mean_recall += recall
            mean_precision += precision 
            mean_f1 += f1
            mean_edit_dist += edit_dist
            mean_exact_match += exact_match

            print("Jaccard = {}".format(jaccard))
            print("Precision = {}".format(precision))
            print("Recall = {}".format(recall))
            print("F1 = {}".format(f1))
            print("edit_dist = {}".format(edit_dist))
            print("EM = {}".format(exact_match))
            cnt += 1

            print("\n\n")


    assert(cnt == N)
    time_end = time.time()

    print("Mean Jaccard = {}".format(mean_jaccard / len(sample_indices)))
    print("Mean Recall = {}".format(mean_recall / len(sample_indices)))
    print("Mean Precision = {}".format(mean_precision / len(sample_indices)))
    print("Mean F1 = {}".format(mean_f1 / len(sample_indices)))
    print("Mean Edit Dist = {}".format(mean_edit_dist / len(sample_indices)))
    print("Mean Exact Match = {}".format(mean_exact_match / len(sample_indices)))
    print("Runtime: {} minutes".format((time_end - time_start) / 60.0))
    


In [ ]:
np.random.seed(33)

N = 1000

sample_indices = np.random.choice(a = np.arange(N), size = 50)

print("Sample Indices = {}".format(sample_indices))

In [ ]:
'''
biassed_contexts = data.get_contexts()[0:N]
unbiassed = data.get_contexts()[N:]

intersection = [(i,context) for (i,context) in enumerate(unbiassed) if context in biassed_contexts]

print(intersection)
'''

In [ ]:
# There is a question overlap of 6, and a context overlap of 0

# data.train_val_cross_section()

# Test Flan T5 L

In [ ]:
model = FlanT5()

## Without Context

In [ ]:
test_model(model, sample_indices, prompts_without_context)

## With Context

In [ ]:
test_model(model, sample_indices, prompts_with_context)

## With Long contexts

In [ ]:
test_model(model, sample_indices, test_prompts = prompts_with_long_context, batch_size = 2)

# Test Flan XL

In [ ]:
del model
model = FlanT5(xl = True)

## Without Context

In [ ]:
test_model(model, sample_indices, prompts_without_context, batch_size = 2)

## With Context

In [ ]:
test_model(model, sample_indices, prompts_with_context, batch_size = 2)

# Test LLama 1B

In [ ]:
del model
model = Llama1B()

## Without Context

In [ ]:
test_model(model, sample_indices, prompts_without_context)

## With Context

In [ ]:
test_model(model, sample_indices, prompts_with_context)

# Test SmolLM1

In [ ]:
#del model # Free Reference from GPU
model = SmolLM(version = 1)

## Without Context

In [ ]:
test_model(model, sample_indices, prompts_without_context)

## With Context

In [ ]:
test_model(model, sample_indices, prompts_with_context)

# Test SmolLM2

In [ ]:
#del model # Free Memory from GPU
model = SmolLM(version = 2)

## Without Context

In [ ]:
test_model(model, sample_indices, prompts_without_context)

## With Context

In [ ]:
test_model(model, sample_indices, prompts_with_context)

# Test Gemma 1B It

In [ ]:
del model # Free Memory from GPU
model = Gemma(it = True, size = 1)

## Without Context

In [ ]:
test_model(model, sample_indices, prompts_without_context)

## With Context

In [ ]:
test_model(model, sample_indices, prompts_with_context)

# Test Gemma 2B It

In [ ]:
del model # Free Memory from GPU
model = Gemma(it = True, size = 2)

## Without Context

In [ ]:
test_model(model, sample_indices, prompts_without_context)

## With Context

In [ ]:
test_model(model, sample_indices, prompts_with_context)

# Test Gemma 2B

In [ ]:
#del model # Free Memory from GPU
model = Gemma(it = False, size = 2)

## Without Context

In [ ]:
test_model(model, sample_indices, prompts_without_context)

## With Context

In [ ]:
test_model(model, sample_indices, prompts_with_context)

# LLama 3B

In [ ]:
del model # Free Memory from GPU
model = Llama3B()

# Without Context

In [ ]:
test_model(model, sample_indices, prompts_without_context)

# With Context

In [ ]:
test_model(model, sample_indices, prompts_with_context)

# Phi3 mini (3.8 B)

In [ ]:
del model 
model = Phi()

# Without Context

In [ ]:
test_model(model, sample_indices, prompts_without_context)

# With Context

In [ ]:
test_model(model, sample_indices, prompts_with_context)

# Test using Llama 3 with API

## NOTE: This worked well in the beginning, but i quickly used the free monthly ammount or i does not work due to high trafiic

In [ ]:
del model
llama3_api = Llama3API()

In [ ]:
prompts = [prompts_with_context[index] for index in sample_indices]
true_answers_slice = [true_answers[index] for index in sample_indices]

for i, prompt in enumerate(prompts):
    print("Prompt: {}\n".format(prompt))
    print("Predicted Answer: {}\n".format(llama3_api._answer_prompt(prompt)))
    print("True Answer: {}\n".format(true_answers_slice[i]))


# Test Using Gemini with API

## Quickly exceeds QUOTA

In [ ]:
gemini_api = GeminiAPI()

In [ ]:
prompts = [prompts_with_context[index] for index in sample_indices]
true_answers_slice = [true_answers[index] for index in sample_indices]

answers = gemini_api.answer(prompts)

for (prompt, answer, answer_true) in zip(prompts, answers, true_answers_slice):
    print("Prompt: {}\nAnswer: {}\nTrue_Answer: {}\n".format(prompt, answer, answer_true))